In [1]:
!wget https://norvig.com/big.txt

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import re
import string
from collections import Counter
from typing import List, Set, Tuple, Dict
import difflib


In [3]:
class SpellChecker:
    def __init__(self, dictionary_file: str = None):
        """
        Initialize the spell checker with a dictionary.

        Args:
            dictionary_file: Path to a text file containing valid words (one per line)
        """
        self.dictionary = set()
        self.word_frequency = Counter()

        if dictionary_file:
            self.load_dictionary(dictionary_file)
            print(f'File Loaded {dictionary_file}')
        else:
            # Default small dictionary for demonstration
            #pass
            self.load_default_dictionary()

    def load_dictionary(self, file_path: str):
        """Load dictionary from a file."""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                words = [line.strip().lower() for line in f if line.strip()]
                self.dictionary = set(words)
                # Simple frequency model (all words have equal weight)
                self.word_frequency = Counter({word: 1 for word in words})
        except FileNotFoundError:
            print(f"Dictionary file {file_path} not found. Using default dictionary.")
            self.load_default_dictionary()

    def load_default_dictionary(self):
        """Load a basic dictionary for demonstration."""
        words = [
            'hello', 'world', 'python', 'programming', 'computer', 'science',
            'algorithm', 'data', 'structure', 'function', 'variable', 'loop',
            'condition', 'class', 'object', 'method', 'string', 'integer',
            'float', 'boolean', 'list', 'dictionary', 'tuple', 'set',
            'import', 'from', 'def', 'return', 'print', 'input', 'output',
            'file', 'read', 'write', 'open', 'close', 'error', 'exception',
            'try', 'except', 'finally', 'with', 'as', 'if', 'else', 'elif',
            'for', 'while', 'break', 'continue', 'pass', 'lambda', 'yield',
            'the', 'and', 'or', 'not', 'in', 'is', 'this', 'that', 'these',
            'those', 'a', 'an', 'to', 'of', 'for', 'with', 'by', 'from',
            'about', 'into', 'through', 'during', 'before', 'after', 'above',
            'below', 'up', 'down', 'out', 'off', 'over', 'under', 'again',
            'further', 'then', 'once', 'here', 'there', 'when', 'where',
            'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more',
            'most', 'other', 'some', 'such', 'only', 'own', 'same', 'so',
            'than', 'too', 'very', 'can', 'will', 'just', 'should', 'now'
        ]
        self.dictionary = set(words)
        self.word_frequency = Counter({word: 1 for word in words})

    def is_valid_word(self, word: str) -> bool:
        """Check if a word exists in the dictionary."""
        return word.lower() in self.dictionary

    def extract_words(self, text: str) -> List[str]:
        """Extract words from text, removing punctuation."""
        # Remove punctuation and split into words
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        return words

    def edit_distance_1(self, word: str) -> Set[str]:
        """Generate all possible words with edit distance of 1."""
        letters = string.ascii_lowercase
        splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]

        # Deletions
        deletes = [L + R[1:] for L, R in splits if R]

        # Transpositions
        transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]

        # Replacements
        replaces = [L + c + R[1:] for L, R in splits if R for c in letters]

        # Insertions
        inserts = [L + c + R for L, R in splits for c in letters]

        return set(deletes + transposes + replaces + inserts)

    def edit_distance_2(self, word: str) -> Set[str]:
        """Generate all possible words with edit distance of 2."""
        return set(e2 for e1 in self.edit_distance_1(word)
                  for e2 in self.edit_distance_1(e1))

    def get_candidates(self, word: str, max_distance: int = 2) -> List[Tuple[str, float]]:
        """
        Get correction candidates for a misspelled word.

        Returns:
            List of tuples (candidate_word, confidence_score)
        """
        if self.is_valid_word(word):
            return [(word, 1.0)]

        candidates = []

        # Try edit distance 1 first
        edit1_candidates = self.edit_distance_1(word)
        valid_edit1 = [w for w in edit1_candidates if self.is_valid_word(w)]

        if valid_edit1:
            # Score based on frequency and similarity
            for candidate in valid_edit1:
                score = self.calculate_score(word, candidate, edit_distance=1)
                candidates.append((candidate, score))

        # If no edit distance 1 candidates and max_distance >= 2, try edit distance 2
        if not candidates and max_distance >= 2:
            edit2_candidates = self.edit_distance_2(word)
            valid_edit2 = [w for w in edit2_candidates if self.is_valid_word(w)]

            for candidate in valid_edit2:
                score = self.calculate_score(word, candidate, edit_distance=2)
                candidates.append((candidate, score))

        # Add candidates based on substring matching
        substring_candidates = self.get_substring_candidates(word)
        candidates.extend(substring_candidates)

        # Remove duplicates and sort by score
        unique_candidates = {}
        for candidate, score in candidates:
            if candidate not in unique_candidates or score > unique_candidates[candidate]:
                unique_candidates[candidate] = score

        # Convert back to list of tuples and sort by score (descending)
        result = [(word, score) for word, score in unique_candidates.items()]
        result.sort(key=lambda x: x[1], reverse=True)

        return result[:10]  # Return top 10 candidates

    def calculate_score(self, original: str, candidate: str, edit_distance: int) -> float:
        """Calculate confidence score for a candidate."""
        # Base score inversely related to edit distance
        base_score = 1.0 / (edit_distance + 1)

        # Frequency bonus
        freq_score = self.word_frequency.get(candidate, 1) / 100.0

        # Length similarity bonus
        len_diff = abs(len(original) - len(candidate))
        len_score = 1.0 / (len_diff + 1)

        # Character overlap bonus
        overlap = len(set(original) & set(candidate))
        overlap_score = overlap / max(len(set(original)), len(set(candidate)))

        return base_score * 0.5 + freq_score * 0.2 + len_score * 0.2 + overlap_score * 0.1

    def get_substring_candidates(self, word: str) -> List[Tuple[str, float]]:
        """Get candidates based on substring matching."""
        candidates = []

        # Find words that contain the input as substring or vice versa
        for dict_word in self.dictionary:
            if word in dict_word or dict_word in word:
                # Calculate similarity score
                similarity = difflib.SequenceMatcher(None, word, dict_word).ratio()
                if similarity > 0.6:  # Threshold for substring candidates
                    score = similarity * 0.3  # Lower score for substring matches
                    candidates.append((dict_word, score))

        return candidates

    def check_text(self, text: str) -> Dict[str, List[Tuple[str, float]]]:
        """
        Check entire text and return corrections for misspelled words.

        Returns:
            Dictionary mapping misspelled words to their correction candidates
        """
        words = self.extract_words(text)
        misspelled = {}

        for word in set(words):  # Use set to avoid checking duplicates
            if not self.is_valid_word(word):
                candidates = self.get_candidates(word)
                if candidates:
                    misspelled[word] = candidates

        return misspelled

    def correct_text(self, text: str, auto_correct: bool = False) -> str:
        """
        Correct text by replacing misspelled words.

        Args:
            text: Input text to correct
            auto_correct: If True, automatically use the best candidate

        Returns:
            Corrected text
        """
        corrections = self.check_text(text)
        corrected_text = text

        for misspelled_word, candidates in corrections.items():
            if candidates:
                if auto_correct:
                    # Use the best candidate
                    best_candidate = candidates[0][0]
                    corrected_text = re.sub(r'\b' + re.escape(misspelled_word) + r'\b',
                                          best_candidate, corrected_text, flags=re.IGNORECASE)
                else:
                    print(f"Misspelled: '{misspelled_word}'")
                    print("Suggestions:")
                    for i, (candidate, score) in enumerate(candidates[:5], 1):
                        print(f"  {i}. {candidate} (confidence: {score:.3f})")
                    print()

        return corrected_text


In [4]:
# Example usage and demonstration
def main():
    # Initialize spell checker
    #spell_checker = SpellChecker(dictionary_file='big.txt')
    spell_checker = SpellChecker()

    print("=== Spell Checker Demo ===\n")

    # Test individual words
    test_words = ["helo", "wrold", "programing", "comuter", "algorithim"]

    print("Individual word corrections:")
    print("-" * 40)
    for word in test_words:
        candidates = spell_checker.get_candidates(word)
        print(f"'{word}' -> {candidates[:3]}")  # Show top 3 candidates

    print("\n" + "="*50 + "\n")

    # Test full text
    test_text = "Helo wrold! This is a smple text with som misspeled words for testng."

    print("Text correction:")
    print("-" * 20)
    print(f"Original: {test_text}")
    print()

    # Check for misspellings
    misspellings = spell_checker.check_text(test_text)
    print("Detected misspellings and suggestions:")
    for word, candidates in misspellings.items():
        print(f"'{word}' -> {[c[0] for c in candidates[:3]]}")

    print()

    # Auto-correct the text
    corrected = spell_checker.correct_text(test_text, auto_correct=True)
    print(f"Auto-corrected: {corrected}")

    print("\n" + "="*50 + "\n")

    # Interactive mode example
    print("Interactive correction example:")
    print("-" * 30)
    interactive_text = "I love programing in pythn!"
    print(f"Text: {interactive_text}")
    print("\nSuggestions:")
    spell_checker.correct_text(interactive_text, auto_correct=False)


In [5]:
main()

=== Spell Checker Demo ===

Individual word corrections:
----------------------------------------
'helo' -> [('hello', 0.45199999999999996)]
'wrold' -> [('world', 0.552)]
'programing' -> [('programming', 0.45199999999999996)]
'comuter' -> [('computer', 0.4395)]
'algorithim' -> [('algorithm', 0.45199999999999996)]


Text correction:
--------------------
Original: Helo wrold! This is a smple text with som misspeled words for testng.

Detected misspellings and suggestions:
'wrold' -> ['world']
'text' -> ['that', 'set']
'som' -> ['some', 'so']
'helo' -> ['hello']
'smple' -> ['tuple']
'words' -> ['world']

Auto-corrected: hello world! This is a tuple that with some misspeled world for testng.


Interactive correction example:
------------------------------
Text: I love programing in pythn!

Suggestions:
Misspelled: 'love'
Suggestions:
  1. over (confidence: 0.444)
  2. loop (confidence: 0.419)
  3. some (confidence: 0.419)
  4. more (confidence: 0.419)
  5. close (confidence: 0.329)

Misspe

In [6]:
sc = SpellChecker(dictionary_file='big.txt')

Dictionary file big.txt not found. Using default dictionary.
File Loaded big.txt


In [7]:
sc.is_valid_word('pthn')

False

In [8]:
sc.edit_distance_1('pthn')

{'apthn',
 'athn',
 'bpthn',
 'bthn',
 'cpthn',
 'cthn',
 'dpthn',
 'dthn',
 'epthn',
 'ethn',
 'fpthn',
 'fthn',
 'gpthn',
 'gthn',
 'hpthn',
 'hthn',
 'ipthn',
 'ithn',
 'jpthn',
 'jthn',
 'kpthn',
 'kthn',
 'lpthn',
 'lthn',
 'mpthn',
 'mthn',
 'npthn',
 'nthn',
 'opthn',
 'othn',
 'pahn',
 'pathn',
 'pbhn',
 'pbthn',
 'pchn',
 'pcthn',
 'pdhn',
 'pdthn',
 'pehn',
 'pethn',
 'pfhn',
 'pfthn',
 'pghn',
 'pgthn',
 'phhn',
 'phn',
 'phthn',
 'phtn',
 'pihn',
 'pithn',
 'pjhn',
 'pjthn',
 'pkhn',
 'pkthn',
 'plhn',
 'plthn',
 'pmhn',
 'pmthn',
 'pnhn',
 'pnthn',
 'pohn',
 'pothn',
 'pphn',
 'ppthn',
 'pqhn',
 'pqthn',
 'prhn',
 'prthn',
 'pshn',
 'psthn',
 'ptahn',
 'ptan',
 'ptbhn',
 'ptbn',
 'ptchn',
 'ptcn',
 'ptdhn',
 'ptdn',
 'ptehn',
 'pten',
 'ptfhn',
 'ptfn',
 'ptghn',
 'ptgn',
 'pth',
 'ptha',
 'pthan',
 'pthb',
 'pthbn',
 'pthc',
 'pthcn',
 'pthd',
 'pthdn',
 'pthe',
 'pthen',
 'pthf',
 'pthfn',
 'pthg',
 'pthgn',
 'pthh',
 'pthhn',
 'pthi',
 'pthin',
 'pthj',
 'pthjn',
 'pthk

In [9]:
sc.is_valid_word('python')

True